## Setup (run every session)
Large artifacts go to Google Drive (`PA3_OUT`); code and small results live in the repo clone. Use the **same GPU type** for every timed run and keep the tab open (Colab Pro has no background execution).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['PA3_OUT'] = '/content/drive/MyDrive/ATML-PA3/outputs'   # checkpoints, caches (persist)
os.environ['PA3_RESULTS'] = '/content/drive/MyDrive/ATML-PA3/results'  # results survive disconnects
os.environ['PA3_COCO'] = '/content/coco'                             # images on fast local disk
os.environ['HF_HOME'] = '/content/hf_cache'
REPO = 'https://github.com/adilawan1/ATML-PA3.git'                 # <-- set


In [ ]:
%cd /content
!test -d ATML-PA3 || git clone $REPO ATML-PA3
%cd /content/ATML-PA3
!git pull -q
!pip -q install -r requirements.txt
# seed Drive results with the committed files (never overwrites newer Drive copies)
!mkdir -p $PA3_RESULTS && rsync -a --ignore-existing results/ $PA3_RESULTS/
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Task 2: SigLIP–Qwen3 VLM
### 1. Data selection (downloads COCO/VQAv2 annotations; skip if `results/task2/data` is committed)

In [ ]:
!test -f $PA3_RESULTS/task2/data/images.json || python -m task2.prepare_data

### 2. Download the selected images and cache SigLIP features (timed separately from training)

In [ ]:
!python -m task2.cache_features

### 3. Train all three stages (re-run to resume)

In [ ]:
!python -m task2.train

### 4. Final evaluation and analysis

In [ ]:
!python -m task2.evaluate --run vis196
!python -m task2.analyze --run vis196

### Optional: 49 visual tokens

In [ ]:
# !python -m task2.train --n_vis 49
# !python -m task2.evaluate --run vis49
# !python -m task2.analyze --run vis196 --compare vis49

## Save small results back to the repo
Commit from Colab (needs a token) or download `results/` and commit locally.

In [ ]:
# copy Drive results into the repo clone, then commit
!rsync -a $PA3_RESULTS/ results/
!git add results && git -c user.name=pa3 -c user.email=pa3@example.com commit -qm "results" || true
# !git push https://<token>@github.com/adilawan1/ATML-PA3.git HEAD